# RMS Zernike Debug 数据分析

用于分析 debug 模式下采集的 WFS 数据：intensity 图像和 deviation 矩阵

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
# 加载数据
data_dir = Path("../data/rms_zernike/2026")
npz_files = list(data_dir.glob("**/wfs_debug_data.npz"))
if npz_files:
    latest_npz = max(npz_files, key=lambda p: p.stat().st_mtime)
    print(f"加载: {latest_npz}")
    data = np.load(latest_npz)
    intensities = data['intensities']
    dev_x = data['dev_x']
    dev_y = data['dev_y']
    print(f"intensities shape: {intensities.shape}")
    print(f"dev_x shape: {dev_x.shape}")
    print(f"dev_y shape: {dev_y.shape}")
else:
    print("未找到 wfs_debug_data.npz 文件")

In [ ]:
def plot_intensity(index):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(intensities[index], cmap='hot', aspect='auto')
    axes[0].set_title(f'Intensity @ epoch {index}')
    axes[0].set_xlabel('X')
    axes[0].set_ylabel('Y')
    
    if dev_x.size > 0 and dev_y.size > 0:
        vmax = np.percentile(np.abs(dev_x[index]), 95)
        im = axes[1].imshow(dev_x[index], cmap='RdBu', vmin=-vmax, vmax=vmax, aspect='auto')
        axes[1].set_title(f'Deviation X @ epoch {index}')
        axes[1].set_xlabel('X')
        axes[1].set_ylabel('Y')
        plt.colorbar(im, ax=axes[1], orientation='horizontal', fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

widgets.interact(
    plot_intensity,
    index=widgets.IntSlider(min=0, max=len(intensities)-1 if intensities.size > 0 else 0, step=1, value=0)
)

In [ ]:
# 统计信息
print("=== Intensity 统计 ===")
print(f"Mean: {np.mean(intensities):.2f}")
print(f"Std: {np.std(intensities):.2f}")
print(f"Max: {np.max(intensities):.2f}")
print(f"Min: {np.min(intensities):.2f}")

if dev_x.size > 0:
    print("\n=== Deviation X 统计 ===")
    print(f"Mean: {np.mean(dev_x):.4f}")
    print(f"Std: {np.std(dev_x):.4f}")
    print(f"Max: {np.max(dev_x):.4f}")
    print(f"Min: {np.min(dev_x):.4f}")

if dev_y.size > 0:
    print("\n=== Deviation Y 统计 ===")
    print(f"Mean: {np.mean(dev_y):.4f}")
    print(f"Std: {np.std(dev_y):.4f}")
    print(f"Max: {np.max(dev_y):.4f}")
    print(f"Min: {np.min(dev_y):.4f}")

In [ ]:
# 时序演化
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].plot([np.mean(i) for i in intensities])
axes[0, 0].set_title('Mean Intensity over epochs')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Mean Intensity')

axes[0, 1].plot([np.std(i) for i in intensities])
axes[0, 1].set_title('Std Intensity over epochs')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Std Intensity')

if dev_x.size > 0:
    axes[1, 0].plot([np.mean(d) for d in dev_x])
    axes[1, 0].set_title('Mean Deviation X over epochs')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Mean Dev X')

if dev_y.size > 0:
    axes[1, 1].plot([np.mean(d) for d in dev_y])
    axes[1, 1].set_title('Mean Deviation Y over epochs')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Mean Dev Y')

plt.tight_layout()
plt.show()

In [ ]:
# 加载并显示优化历史
import pickle
pkl_files = list(data_dir.glob("**/*.pkl"))
if pkl_files:
    latest_pkl = max(pkl_files, key=lambda p: p.stat().st_mtime)
    print(f"加载历史: {latest_pkl}")
    with open(latest_pkl, 'rb') as f:
        records = pickle.load(f)
    
    rms_history = records.get_sublist()
    plt.figure(figsize=(10, 4))
    plt.plot(rms_history)
    plt.title('RMS over epochs')
    plt.xlabel('Epoch')
    plt.ylabel('RMS')
    plt.grid(True)
    plt.show()
    print(f"最终 RMS: {rms_history[-1]:.4f}")
    print(f"最佳 RMS: {min(rms_history):.4f}")